In [1]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 6.76569194e-02  6.34959415e-02  4.87130806e-02  7.93049484e-02
   3.74481045e-02  2.65281182e-03  3.93749401e-02 -7.09844474e-03
   5.93614615e-02  3.15370038e-02  6.00980595e-02 -5.29052690e-02
   4.06068154e-02 -2.59308740e-02  2.98427939e-02  1.12690637e-03
   7.35148638e-02 -5.03818318e-02 -1.22386597e-01  2.37028114e-02
   2.97265425e-02  4.24768664e-02  2.56337449e-02  1.99515442e-03
  -5.69190457e-02 -2.71598250e-02 -3.29035148e-02  6.60249218e-02
   1.19007207e-01 -4.58791330e-02 -7.26214126e-02 -3.25840451e-02
   5.23412861e-02  4.50553298e-02  8.25300720e-03  3.67024392e-02
  -1.39415609e-02  6.53918684e-02 -2.64272150e-02  2.06383236e-04
  -1.36643508e-02 -3.62810828e-02 -1.95043813e-02 -2.89738141e-02
   3.94270159e-02 -8.84090737e-02  2.62426236e-03  1.36713963e-02
   4.83063012e-02 -3.11566219e-02 -1.17329165e-01 -5.11690676e-02
  -8.85288119e-02 -2.18963195e-02  1.42986281e-02  4.44167927e-02
  -1.34815322e-02  7.43392557e-02  2.66382676e-02 -1.98762883e-02
   1.79191

In [28]:
sentence1 = "Hello little man"
sentence2 = ["Hello little arsch"]

embeddings = model.encode(sentence1)
embeddings2 = model.encode(sentence2)
emb = embeddings - embeddings2
#print(emb)

In [30]:
tokens_info = model.tokenizer(sentence1)
token_ids = tokens_info['input_ids']
print(token_ids)
tokens = model.tokenizer.decode(token_ids)
print(tokens)

[101, 7592, 2210, 2158, 102]
[CLS] hello little man [SEP]


In [53]:
import numpy as np
from scipy.optimize import least_squares

def calc_forward(token_ids, model):
  token_ids_list = []
  for tok in token_ids:
    token_ids_list.append(int(tok))

  tokens = model.tokenizer.decode(token_ids_list)
  embeddings = model.encode(tokens)
  return embeddings

def calc_formular(kQ, target_emb, model):
  return calc_forward(kQ, model) - target_emb

def optimize_function(func, q0, minToken, maxToken, target, model):
  res = least_squares(func, q0, bounds=(minToken, maxToken), args=(target, model))
  return res.x

In [54]:
test_sentence = "Hello my Friend"
wished_embeddings = model.encode(sentence1)

max_id = model.tokenizer.vocab_size - 1
min_id = 0
max_answer_len = 20

q0 = [] #liste mit gewünschter satztlänge füllen

for i in range(max_answer_len):
  q0.append(0)

print(len(q0))

result = optimize_function(calc_formular, q0, min_id, max_id,target=wished_embeddings, model=model)
token_ids_list = []
for tok in result:
  token_ids_list.append(int(tok))

tokens = model.tokenizer.decode(token_ids_list)
print(tokens)

20
[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [56]:
test = "[PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]"
test_sentence = "Hello my Friend"
emb = model.encode(test)
emb2 = model.encode(test_sentence)
#print(emb-emb2)

In [ ]:
#diese idee funktioniert nicht soo wie ich es geplant hatte. Das Problem war der least_squares algo, welcher mein Vorhaben nicht erlaubt. Eine andere minimieren der funktion ist von not